## Reviews Table - PySpark Cleaning

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, to_date
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("ReviewsCleaning").getOrCreate()

df_reviews = spark.read.csv("../Coursework_dataset/reviews.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("../Coursework_dataset/orders.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:25:39 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:25:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:25:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/14 18:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/14 18:25:41 WARN Utils: Servic

### Preview the reviews table

In [2]:
df_reviews.show(5)

+----------+----------+----------+------------+-----------+--------------------+-------------------+
| review_id|  order_id|product_id|review_score|review_date|         review_text|review_length_chars|
+----------+----------+----------+------------+-----------+--------------------+-------------------+
|rev_000001|ord_200369|prod_01620|         3.0| 2024-01-06|Average quality. ...|                214|
|rev_000002|ord_306978|prod_01067|         4.0| 2023-12-19|Amazing product! ...|                146|
|rev_000003|ord_078396|prod_04954|         5.0| 2022-10-29|Fantastic purchas...|                213|
|rev_000004|ord_231351|prod_00296|         4.0| 2022-04-26|Perfect for my ne...|                185|
|rev_000005|ord_244947|prod_02596|         5.0| 2021-12-02|Amazing product! ...|                149|
+----------+----------+----------+------------+-----------+--------------------+-------------------+
only showing top 5 rows


### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_reviews.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_reviews.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,review_id,280000
1,order_id,280000
2,product_id,280000
3,review_score,277000
4,review_date,280000
5,review_text,280000
6,review_length_chars,280000


### Check which columns have null values

In [4]:
null_counts = df_reviews.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_reviews.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,review_id,0
1,order_id,0
2,product_id,0
3,review_score,3000
4,review_date,0
5,review_text,0
6,review_length_chars,0


### Fix 1: Fill null review_score with 0
A missing score is treated as 0 (no rating given).

In [5]:
df_reviews = df_reviews.withColumn(
    "review_score",
    when(col("review_score").isNull(), lit(0.0))
    .otherwise(col("review_score"))
)

print("Null review_score remaining:", df_reviews.filter(col("review_score").isNull()).count())

Null review_score remaining: 0


### Fix 2: Cap review_score at 5.0
The valid score range is 0 to 5. Any score of 6.0 is a data error and should be corrected to 5.0.

In [6]:
# Show how many rows have a score of 6.0 before fixing
print("Rows with score 6.0:", df_reviews.filter(col("review_score") == 6.0).count())

df_reviews = df_reviews.withColumn(
    "review_score",
    when(col("review_score") == 6.0, lit(5.0))
    .otherwise(col("review_score"))
)

print("Rows with score 6.0 after fix:", df_reviews.filter(col("review_score") == 6.0).count())

Rows with score 6.0: 600
Rows with score 6.0 after fix: 0


### Fix 3: Cast review_score to Double
Ensure the score is stored as a floating-point number (Double), not an integer or string.

In [7]:
df_reviews = df_reviews.withColumn("review_score", col("review_score").cast(DoubleType()))

# Confirm the score distribution looks correct
df_reviews.groupBy("review_score").count().orderBy("review_score").show()

+------------+-----+
|review_score|count|
+------------+-----+
|         0.0| 3000|
|         1.0|22272|
|         2.0|31765|
|         3.0|49581|
|         4.0|89007|
|         5.0|84375|
+------------+-----+



### Fix 4: Convert review_date to a proper date type
Stored as a string - we cast it to DateType so date comparisons work correctly.

In [8]:
df_reviews = df_reviews.withColumn("review_date", to_date(col("review_date")))

print("Unparseable review_date values:", df_reviews.filter(col("review_date").isNull()).count())

Unparseable review_date values: 0


### Check 5: Find reviews where the review date is before the order date
A review cannot happen before the order was placed. We join reviews with orders to find these timeline violations.

In [9]:
# Convert order_date to DateType as well for comparison
df_orders = df_orders.withColumn("order_date", to_date(col("order_date")))

# Join reviews with orders on order_id (Merge Join equivalent)
df_joined = df_reviews.join(
    df_orders.select("order_id", "order_date"),
    on="order_id",
    how="left"
)

# Find rows where the review was written before the order was placed
timeline_violations = df_joined.filter(
    col("review_date") < col("order_date")
)

print("Reviews with date before order date:", timeline_violations.count())
timeline_violations.select("review_id", "order_id", "review_date", "order_date").show(10)

Reviews with date before order date: 180
+----------+----------+-----------+----------+
| review_id|  order_id|review_date|order_date|
+----------+----------+-----------+----------+
|rev_001594|ord_149926| 2020-07-13|2021-01-08|
|rev_004248|ord_147311| 2023-10-07|2023-12-13|
|rev_004312|ord_130062| 2022-04-25|2022-05-29|
|rev_005060|ord_329279| 2022-11-15|2023-02-04|
|rev_005227|ord_008064| 2021-07-07|2021-12-20|
|rev_006403|ord_152888| 2019-04-14|2019-10-10|
|rev_006876|ord_099563| 2022-07-30|2022-10-23|
|rev_008501|ord_320066| 2023-06-11|2023-11-04|
|rev_008799|ord_310300| 2021-01-10|2021-04-21|
|rev_010508|ord_303947| 2023-07-29|2023-11-17|
+----------+----------+-----------+----------+
only showing top 10 rows


### Check the schema to confirm data types

In [10]:
df_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- review_score: double (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_length_chars: integer (nullable = true)



### Final check: Verify null counts after all cleaning

In [11]:
final_nulls = df_reviews.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_reviews.columns
])
final_null_df = final_nulls.toPandas().T.reset_index()
final_null_df.columns = ['Column Name', 'Null Count']
final_null_df

,Column Name,Null Count
0,review_id,0
1,order_id,0
2,product_id,0
3,review_score,0
4,review_date,0
5,review_text,0
6,review_length_chars,0


### Export the cleaned reviews table

In [12]:
df_reviews.coalesce(1).write.csv("../cleaned_dataset/reviews_cleaned.csv", header=True, mode="overwrite")
print("Reviews table exported successfully to 'cleaned_dataset/reviews_cleaned.csv'")

Reviews table exported successfully to 'cleaned_dataset/reviews_cleaned.csv'
